```markdown
# Question 1: Knowledge Distillation Framework (50 Marks)

Develop an end-to-end knowledge distillation framework that transfers learned representations from a large teacher network into lightweight student networks.

### Requirements:
1. **Teacher Model**: Implement and train a teacher model using **ResNet-50 on CIFAR-100**.
2. **Student Architectures**: Implement three lightweight student architectures:
   - MobileNetV2
   - SqueezeNet
   - EfficientNet-Lite0
3. **Distillation Loss Support**:
   - Soft-target knowledge distillation using teacher predictions.
   - Temperature-based softmax.
   - Combined hard-label and soft-label training.
   - Feature-based distillation using intermediate feature representations.
4. **Student Training**: Train all three student architectures using the implemented distillation framework.
5. **Baseline Pipelines**: Implement a separate baseline training pipeline for the three student architectures without knowledge distillation.
6. **Unified Inference**: Develop a unified inference pipeline supporting all trained student models.
7. **Export & CPU Inference**: Export the trained student models into ONNX format and implement CPU-based inference using ONNX Runtime.
```

In [1]:
# =====================================================================
# STEP 1: Dependencies, Imports & Dataset Preparation (CIFAR-100)
# =====================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import os

# Hyperparameter: Select processing unit ('cuda' for GPU acceleration, else 'cpu')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Transform Hyperparameters:
# - RandomCrop: Pad size = 4 (adds reflection-padded margins before cropping to 32x32)
# - Normalize: Predefined mean and standard deviation vectors calculated over CIFAR-100 channel arrays
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(), # Flips images horizontally with a 50% probability
    transforms.ToTensor(),             # Converts images to range [0.0, 1.0] PyTorch Tensors
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# Dataset loader instantiation
train_set = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=train_transform
)

test_set = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=test_transform
)

# Dataloader Hyperparameters:
# - batch_size = 128: Number of sample images processed in a single training iteration
# - num_workers = 2: Number of parallel CPU subprocesses dedicated to data loading tasks
# - pin_memory = True: Automatically copies Tensors into CUDA pinned memory for faster GPU transfer
batch_size = 128
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataset loaded: {len(train_set)} training samples, {len(test_set)} testing samples.")

Using device: cuda


100%|██████████| 169M/169M [24:32<00:00, 115kB/s]  


Dataset loaded: 50000 training samples, 10000 testing samples.


In [2]:
# =====================================================================
# STEP 2: Defining Teacher and Student Architectures for CIFAR-100
# =====================================================================

import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. Teacher Model: ResNet-50 ---
def get_resnet50_teacher(pretrained=False):
    """
    Instantiates a ResNet-50 network and modifies key layers to process 32x32 images effectively.

    Parameters:
        pretrained (bool): If True, downloads pre-trained ImageNet weights (not recommended for custom setups).
    """
    model = models.resnet50(pretrained=pretrained)

    # Downsampling Adjustment:
    # Change conv1 from 7x7 stride=2 to 3x3 stride=1 to prevent high loss of spatial detail on 32x32 images
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # Bypass MaxPool to preserve spatial resolution in the early stages of the deep network
    model.maxpool = nn.Identity()

    # Classifier Layer Adjustment:
    # Set final Linear out_features dimension to 100 to map to CIFAR-100 target labels
    model.fc = nn.Linear(model.fc.in_features, 100)
    return model


# --- 2. Student 1: MobileNetV2 ---
def get_mobilenet_v2_student():
    """
    Instantiates a standard MobileNetV2 model adapted for lightweight CIFAR-100 classification.
    """
    model = models.mobilenet_v2(pretrained=False)

    # Change first convolution stride from 2 to 1 to protect initial image dimensions
    model.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)

    # Map features directly to the target classification size (100 labels)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 100)
    return model


# --- 3. Student 2: SqueezeNet ---
def get_squeezenet_student():
    """
    Instantiates SqueezeNet 1_1 modified to target 100 unique classes.
    """
    model = models.squeezenet1_1(pretrained=False)

    # Retain input image dimensions on first feature layer convolution
    model.features[0] = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)

    # Update convolutional classification layer outputs from 1000 to 100
    model.classifier[1] = nn.Conv2d(512, 100, kernel_size=1, stride=1)
    model.num_classes = 100
    return model


# --- 4. Student 3: EfficientNet-Lite0 ---
def get_efficientnet_lite0_student():
    """
    Instantiates an EfficientNet-B0 backbone serving as our lightweight 'Lite0' representation.
    """
    model = models.efficientnet_b0(pretrained=False)

    # Set stride parameter to 1 to scale input processing safely for CIFAR-100's small dimensions
    model.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False)

    # Map representation to 100 classification outputs
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 100)
    return model

# Model instance setups
teacher = get_resnet50_teacher().to(device)
mobilenet_s = get_mobilenet_v2_student().to(device)
squeezenet_s = get_squeezenet_student().to(device)
efficientnet_s = get_efficientnet_lite0_student().to(device)

print("Teacher & Student models instantiated and configured for CIFAR-100 successfully!")

Using device: cuda


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Teacher & Student models instantiated and configured for CIFAR-100 successfully!


In [3]:
# =====================================================================
# STEP 3: Knowledge Distillation Loss Framework (Soft + Hard + Feature)
# =====================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class DistillationLoss(nn.Module):
    """
    A unified PyTorch loss module combining hard targets and soft targets with temperature scaling.
    """
    def __init__(self, temperature=4.0, alpha=0.3, beta=0.7, feature_weight=0.1):
        """
        Parameters:
            temperature (float): Controls softening of probability distributions (higher values = softer target curves).
            alpha (float): Scaling factor for traditional hard labels (Cross-Entropy loss).
            beta (float): Scaling factor for soft target matching (KL-Divergence loss).
            feature_weight (float): Multiplier scaling intermediate feature map MSE alignments.
        """
        super(DistillationLoss, self).__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.beta = beta
        self.feature_weight = feature_weight

        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')
        self.mse_loss = nn.MSELoss()

    def forward(self, student_logits, teacher_logits, ground_truth,
                student_features=None, teacher_features=None, feature_projector=None):
        """
        Executes the loss combination math for each training batch step.
        """
        # 1. Hard Classification loss (Cross-Entropy between student output and labels)
        loss_hard = self.ce_loss(student_logits, ground_truth)

        # 2. Soft distillation loss using KL divergence on softened probability outputs
        soft_student = F.log_softmax(student_logits / self.temperature, dim=1)
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=1)

        # Multiply by T^2 to preserve comparable gradient magnitudes when scaling temperatures
        loss_soft = self.kl_loss(soft_student, soft_teacher) * (self.temperature ** 2)

        # 3. Optional intermediate feature extraction mapping alignments
        loss_features = torch.tensor(0.0).to(student_logits.device)
        if student_features is not None and teacher_features is not None:
            if feature_projector is not None:
                projected_student_features = feature_projector(student_features)
            else:
                projected_student_features = student_features
            loss_features = self.mse_loss(projected_student_features, teacher_features)

        # Weighted sum composition of the multi-task loss
        total_loss = (self.alpha * loss_hard) + (self.beta * loss_soft) + (self.feature_weight * loss_features)
        return total_loss, loss_hard, loss_soft, loss_features

print("Unified Distillation Loss module implemented and ready for training!")

Unified Distillation Loss module implemented and ready for training!


In [4]:
# =====================================================================
# STEP 4: Standard Training & Evaluation Loops + Train ResNet-50 Teacher
# =====================================================================

import torch
import torch.optim as optim
import time

def evaluate_accuracy(model, data_loader, device):
    """
    Evaluates model accuracy using a simple argmax match iteration.

    Parameters:
        model (nn.Module): The target neural network configuration.
        data_loader (DataLoader): The evaluation pipeline (validation or test dataset).
        device (torch.device): CPU or CUDA target.
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    accuracy = 100.0 * correct / total
    return accuracy

def train_standard_model(model, train_loader, val_loader, epochs, lr, device):
    """
    Trains a standard neural network configuration without distillation hooks.

    Parameters:
        epochs (int): Total times to iterate through the complete dataset.
        lr (float): Optimizer starting step magnitude (learning rate).
        weight_decay (float): L2 regularization constraint preventing large parameter weights.
    """
    model = model.to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Scheduler: Gradually decays learning rate down following a cosine curve trajectory
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"--- Starting Training on {device} ---")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        scheduler.step()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = 100.0 * correct / total
        val_acc = evaluate_accuracy(model, val_loader, device)
        epoch_time = time.time() - start_time

        print(f"Epoch [{epoch+1}/{epochs}] ({epoch_time:.1f}s) - Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - Val Acc: {val_acc:.2f}%")

    return model

# Train our large Teacher model using 5 epochs as a demonstration baseline.
print("Training ResNet-50 Teacher on CIFAR-100...")
teacher = train_standard_model(teacher, train_loader, test_loader, epochs=5, lr=1e-3, device=device)

Training ResNet-50 Teacher on CIFAR-100...
--- Starting Training on cuda ---
Epoch [1/5] (195.2s) - Loss: 4.0019 - Train Acc: 8.80% - Val Acc: 14.10%
Epoch [2/5] (204.6s) - Loss: 3.1660 - Train Acc: 21.11% - Val Acc: 25.66%
Epoch [3/5] (205.1s) - Loss: 2.5459 - Train Acc: 33.27% - Val Acc: 37.28%
Epoch [4/5] (204.5s) - Loss: 2.0564 - Train Acc: 43.96% - Val Acc: 42.57%
Epoch [5/5] (205.2s) - Loss: 1.6769 - Train Acc: 52.98% - Val Acc: 52.47%


In [5]:
# =====================================================================
# STEP 5: Baseline and Distillation Student Training Loops
# =====================================================================

import torch
import torch.optim as optim
import time

def train_distilled_student(student, teacher, train_loader, val_loader, epochs, lr, device, distillation_loss_fn):
    """
    Applies the distillation loss framework to guide a lightweight student using a trained teacher.

    Parameters:
        student (nn.Module): The lightweight student network being optimized.
        teacher (nn.Module): The frozen, high-accuracy teacher providing soft targets.
        epochs (int): Number of target passes across the dataset.
        lr (float): Learning rate value.
        distillation_loss_fn (DistillationLoss): Our unified custom loss configuration.
    """
    teacher.eval() # Ensure the teacher is running in evaluation mode (no dropout/batchnorm updates)
    student = student.to(device)

    optimizer = optim.Adam(student.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"--- Starting Distillation Training on {device} ---")
    for epoch in range(epochs):
        student.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()

            # Extract teacher outputs with gradients disabled
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Compute forward pass outputs on student model
            student_logits = student(inputs)

            # Execute soft target loss matching calculations
            loss, loss_hard, loss_soft, _ = distillation_loss_fn(
                student_logits=student_logits,
                teacher_logits=teacher_logits,
                ground_truth=targets
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = student_logits.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        scheduler.step()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = 100.0 * correct / total
        val_acc = evaluate_accuracy(student, val_loader, device)
        epoch_time = time.time() - start_time

        print(f"Epoch [{epoch+1}/{epochs}] ({epoch_time:.1f}s) - Distill Loss: {epoch_loss:.4f} - Train Acc: {epoch_acc:.2f}% - Val Acc: {val_acc:.2f}%")

    return student

# Hyperparameters for Distillation:
# - temperature = 4.0: Logit scaling to soften target classification spikes
# - alpha = 0.3: Weight assigned to direct classification Loss
# - beta = 0.7: High focus scaling directed to soft KL Divergence updates
loss_distill_config = DistillationLoss(temperature=4.0, alpha=0.3, beta=0.7)

print("Baseline and distillation training loops are successfully declared!")

Baseline and distillation training loops are successfully declared!


In [6]:
# =====================================================================
# STEP 6: Train Baseline and Distilled Student Models
# =====================================================================

# This serves as our comparative verification block.
# We train our MobileNetV2 under both training regimes for 10 epochs to evaluate benefits.

print("=== Training MobileNetV2: Baseline (No Distillation) ===")
mobilenet_baseline = get_mobilenet_v2_student().to(device)
mobilenet_baseline = train_standard_model(
    model=mobilenet_baseline,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: Set to 10 epochs for efficient validation
    lr=1e-3,
    device=device
)

print("\n=== Training MobileNetV2: Distilled ===")
mobilenet_distilled = get_mobilenet_v2_student().to(device)
mobilenet_distilled = train_distilled_student(
    student=mobilenet_distilled,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: Set to 10 epochs for efficient validation
    lr=1e-3,
    device=device,
    distillation_loss_fn=loss_distill_config
)

=== Training MobileNetV2: Baseline (No Distillation) ===
--- Starting Training on cuda ---
Epoch [1/10] (26.2s) - Loss: 4.1009 - Train Acc: 6.97% - Val Acc: 11.62%
Epoch [2/10] (26.0s) - Loss: 3.6027 - Train Acc: 13.94% - Val Acc: 16.84%
Epoch [3/10] (26.1s) - Loss: 3.3096 - Train Acc: 18.87% - Val Acc: 20.92%
Epoch [4/10] (25.9s) - Loss: 3.0696 - Train Acc: 23.29% - Val Acc: 26.15%
Epoch [5/10] (25.9s) - Loss: 2.8413 - Train Acc: 27.69% - Val Acc: 30.33%
Epoch [6/10] (25.8s) - Loss: 2.6425 - Train Acc: 31.59% - Val Acc: 33.69%
Epoch [7/10] (25.8s) - Loss: 2.4725 - Train Acc: 35.07% - Val Acc: 36.25%
Epoch [8/10] (25.8s) - Loss: 2.3285 - Train Acc: 38.28% - Val Acc: 38.87%
Epoch [9/10] (25.9s) - Loss: 2.2130 - Train Acc: 40.62% - Val Acc: 40.26%
Epoch [10/10] (25.8s) - Loss: 2.1471 - Train Acc: 42.41% - Val Acc: 40.92%

=== Training MobileNetV2: Distilled ===
--- Starting Distillation Training on cuda ---
Epoch [1/10] (87.4s) - Distill Loss: 4.6405 - Train Acc: 9.18% - Val Acc: 16.82%


In [7]:
# =====================================================================
# STEP 7: Unified Inference Pipeline
# =====================================================================

import torch
import numpy as np

def run_unified_inference(model, input_tensor, device):
    """
    Runs standard inference over target evaluation tensors for any student framework model type.

    Parameters:
        model (nn.Module): Any custom student network model structure.
        input_tensor (torch.Tensor): Preprocessed input tensor of shape [Batch, Channels, Height, Width].
        device (torch.device): Destination target device classification.
    """
    model.eval()
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        logits = model(input_tensor)

        # SqueezeNet outputs convolutional shapes [B, 100, 1, 1]; flatten if detected
        if len(logits.shape) == 4:
            logits = logits.view(logits.size(0), -1)

        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        predicted_classes = torch.argmax(logits, dim=1).cpu().tolist()

    return predicted_classes, probabilities

# Inference Verification:
# Generates a randomized evaluation tensor representing a single CIFAR image block [1, 3, 32, 32]
dummy_input = torch.randn(1, 3, 32, 32)
predictions, probs = run_unified_inference(mobilenet_distilled, dummy_input, device)
print(f"Unified inference call successful! Predicted class: {predictions[0]} with confidence: {probs[0][predictions[0]]:.4f}")

Unified inference call successful! Predicted class: 39 with confidence: 0.9957


In [8]:
# =====================================================================
# STEP 8: Exporting to ONNX and Running CPU Inference with ONNX Runtime
# =====================================================================

!pip install -q onnx onnxruntime onnxscript

import onnx
import onnxruntime as ort
import numpy as np

def export_student_to_onnx(model, save_path="student.onnx"):
    """
    Saves a trained PyTorch model as a standardized serialized graph output (ONNX file).

    Parameters:
        model (nn.Module): Model selected for export.
        save_path (str): Target destination file name.
    """
    model.eval()
    # Provide dummy tensor input mimicking the shape expected by the model inputs
    dummy_input = torch.randn(1, 3, 32, 32, device=device)

    # Export with Opset 18 to handle complex architectures smoothly
    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        export_params=True,             # Embed parameter weights directly inside the exported file
        opset_version=18,
        do_constant_folding=True,       # Compresses constant mathematical steps for runtime savings
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}} # Allows variable batch sizes at runtime
    )
    print(f"Model successfully exported to ONNX format at: {save_path}")

def run_onnx_cpu_inference(onnx_model_path, input_tensor):
    """
    Runs inferences on an exported ONNX model on CPU using ONNX Runtime.
    """
    session = ort.InferenceSession(onnx_model_path, providers=['CPUExecutionProvider'])
    onnx_inputs = {session.get_inputs()[0].name: input_tensor.cpu().numpy()}
    onnx_outputs = session.run(None, onnx_inputs)
    logits = onnx_outputs[0]

    if len(logits.shape) == 4:
        logits = logits.reshape(logits.shape[0], -1)

    predicted_class = np.argmax(logits, axis=1)[0]
    return predicted_class

# Export evaluation step
export_student_to_onnx(mobilenet_distilled, "mobilenet_distilled.onnx")

test_tensor = torch.randn(1, 3, 32, 32)
onnx_pred = run_onnx_cpu_inference("mobilenet_distilled.onnx", test_tensor)
print(f"ONNX Runtime CPU Inference completed successfully! Predicted class ID: {onnx_pred}")

/tmp/ipykernel_49/2822745361.py:24: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully exported to ONNX format at: mobilenet_distilled.onnx
ONNX Runtime CPU Inference completed successfully! Predicted class ID: 39


```markdown
## Question 2: Automated Neural Architecture Search (NAS) Framework

In this section, we implement an automated NAS framework to discover lightweight CNNs suitable for resource-constrained edge devices.

### Step 1: Define the Search Space
We define a searchable building block that supports:
- Depthwise Separable Convolutions vs. Standard Convolutions
- Multiple Kernel Sizes (`3x3`, `5x5`)
- Variable activations (`ReLU`, `SiLU` aka Swish, `LeakyReLU`)
- Variable filters/channels
```

In [9]:
# =====================================================================
# STEP 9: Define the Searchable CNN Space for NAS
# =====================================================================

import torch
import torch.nn as nn

class DepthwiseSeparableConv(nn.Module):
    """
    A lightweight convolutional alternative splitting regular convolutions into depthwise and pointwise layers.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, activation_fn=nn.ReLU):
        """
        Parameters:
            in_channels (int): Incoming spatial channels.
            out_channels (int): Destination channel depth.
            kernel_size (int): Receptive field size (3x3 or 5x5).
            activation_fn (class): The target activation class.
        """
        super(DepthwiseSeparableConv, self).__init__()
        padding = kernel_size // 2

        # 1. Depthwise: Grouped convolution matching in_channels, reducing parameter footprints
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size=kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=False
        )
        # 2. Pointwise: 1x1 convolution merging channel representations into out_channels
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = activation_fn()

    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))

class SearchableConvBlock(nn.Module):
    """
    A searchable meta-block configuring block styles dynamically according to choice dictionaries.
    """
    def __init__(self, in_channels, out_channels, choice_dict):
        """
        Parameters:
            choice_dict (dict): Sub-configuration mapped to the current block containing 'conv_type', 'kernel_size', 'activation'.
        """
        super(SearchableConvBlock, self).__init__()
        conv_type = choice_dict.get('conv_type', 'standard')  # Options: 'standard' or 'depthwise_separable'
        kernel_size = choice_dict.get('kernel_size', 3)
        act_name = choice_dict.get('activation', 'relu')      # Options: 'relu', 'silu', 'leaky_relu'

        # Dynamically map activation names to activation modules
        if act_name == 'silu':
            activation_fn = nn.SiLU
        elif act_name == 'leaky_relu':
            activation_fn = lambda: nn.LeakyReLU(0.1)
        else:
            activation_fn = nn.ReLU

        padding = kernel_size // 2

        # Construct either a standard convolutional block or a depthwise separable block
        if conv_type == 'depthwise_separable':
            self.conv = DepthwiseSeparableConv(in_channels, out_channels, kernel_size, stride=1, activation_fn=activation_fn)
        else:
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False),
                nn.BatchNorm2d(out_channels),
                activation_fn()
            )

    def forward(self, x):
        return self.conv(x)

print("Search space blocks and Depthwise Separable classes declared successfully!")

Search space blocks and Depthwise Separable classes declared successfully!


In [10]:
# =====================================================================
# STEP 10: Implement Searchable CNN Architecture and NAS Evaluation
# =====================================================================

import torch
import torch.nn as nn
import random

class SearchableNet(nn.Module):
    """
    A flexible architecture template instantiated directly from searchable configuration files.
    """
    def __init__(self, config, num_classes=100):
        """
        Parameters:
            config (dict): Target recipe dict defining 'depth', 'channels', 'kernels', 'conv_types', 'activation'.
        """
        super(SearchableNet, self).__init__()

        depth = config['depth']            # Search Parameter: Number of sequential blocks
        channels = config['channels']      # Search Parameter: Channel width per block
        kernels = config['kernels']        # Search Parameter: Kernel size per block
        conv_types = config['conv_types']  # Search Parameter: Conv block style per block
        activation = config['activation']  # Search Parameter: Global activation function choice

        layers = []
        in_c = 3

        for i in range(depth):
            out_c = channels[i]
            choice_dict = {
                'conv_type': conv_types[i],
                'kernel_size': kernels[i],
                'activation': activation
            }
            layers.append(SearchableConvBlock(in_c, out_c, choice_dict))
            in_c = out_c

            # Periodically downsample features to manage parameters and keep computations fast
            if i in [1, 3] or i == (depth - 1):
                layers.append(nn.MaxPool2d(2, 2))

        self.backbone = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_c, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- SEARCH SPACE RANGES ---
search_space = {
    'depths': [3, 4, 5],                       # Selectable range for network layers
    'channels': [16, 32, 64, 128],             # Selectable filter configurations
    'kernels': [3, 5],                         # Selectable kernel sizes
    'conv_types': ['standard', 'depthwise_separable'],
    'activations': ['relu', 'silu', 'leaky_relu']
}

def sample_random_architecture():
    """
    Randomly samples an architectural configuration within the defined search space.
    """
    depth = random.choice(search_space['depths'])
    config = {
        'depth': depth,
        'channels': [random.choice(search_space['channels']) for _ in range(depth)],
        'kernels': [random.choice(search_space['kernels']) for _ in range(depth)],
        'conv_types': [random.choice(search_space['conv_types']) for _ in range(depth)],
        'activation': random.choice(search_space['activations'])
    }
    return config

random_cfg = sample_random_architecture()
demo_net = SearchableNet(random_cfg).to(device)
print(f"Random Configuration Sampled: {random_cfg}")
print(f"Total network parameters: {count_parameters(demo_net):,}")

Random Configuration Sampled: {'depth': 4, 'channels': [128, 128, 64, 16], 'kernels': [3, 3, 5, 3], 'conv_types': ['depthwise_separable', 'standard', 'depthwise_separable', 'depthwise_separable'], 'activation': 'silu'}
Total network parameters: 163,231


In [11]:
# =====================================================================
# STEP 11: Automated NAS Search Controller (Random Search Evaluation)
# =====================================================================

import torch
import torch.optim as optim
from torch.utils.data import Subset
import time

def evaluate_nas_candidate(config, device, train_loader, val_loader, max_train_samples=5000):
    """
    Trains a candidate network for a very short duration (1 proxy epoch on a small subset of data)
    to quickly score its learning ability and measure parameter size.

    Parameters:
        config (dict): The sampled architecture recipe (depth, channels, kernels, etc.)
        device (torch.device): Device to run training on (CUDA/GPU or CPU)
        train_loader (DataLoader): Full training dataset loader
        val_loader (DataLoader): Validation/Test dataset loader
        max_train_samples (int): Max images to use for proxy training (speeds up search)
    """
    # 1. Instantiate searchable model template with the current configuration
    model = SearchableNet(config).to(device)
    param_count = count_parameters(model)

    # 2. Slice dataset into a smaller subset for fast proxy evaluation
    subset_indices = list(range(min(max_train_samples, len(train_loader.dataset))))
    subset_train_dataset = Subset(train_loader.dataset, subset_indices)
    subset_train_loader = DataLoader(
        subset_train_dataset,
        batch_size=64,       # Hyperparameter: Mini-batch size for quick evaluation
        shuffle=True
    )

    # 3. Quick learning rate & optimizer setup
    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-3              # Hyperparameter: Learning rate for training candidates
    )
    criterion = nn.CrossEntropyLoss()

    # 4. Single-epoch proxy training loop
    model.train()
    for inputs, targets in subset_train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

    # 5. Fast model evaluation on the test set
    val_acc = evaluate_accuracy(model, val_loader, device)

    # 6. Multi-objective scoring
    # Score = Accuracy / Params (in millions) with a tiny constant offset to protect against division by zero
    params_in_millions = max(param_count, 1) / 1e6
    nas_score = val_acc / (params_in_millions + 0.01)

    return param_count, val_acc, nas_score

# --- SEARCH CONFIGURATION & EXECUTION ---
num_search_trials = 5  # Hyperparameter: Number of distinct networks to sample and evaluate
best_config = None
best_score = -1.0
nas_results = []

print(f"=== Starting Neural Architecture Search (Trials = {num_search_trials}) ===\n")
for trial in range(num_search_trials):
    config = sample_random_architecture()
    print(f"Trial {trial+1}: Evaluating configuration...\n{config}")

    try:
        params, acc, score = evaluate_nas_candidate(config, device, train_loader, test_loader)
        print(f"-> Results: Params = {params:,} | Validation Acc = {acc:.2f}% | Multi-objective Score = {score:.4f}\n")

        nas_results.append({
            'config': config,
            'params': params,
            'accuracy': acc,
            'score': score
        })

        if score > best_score:
            best_score = score
            best_config = config
    except Exception as e:
        print(f"Failed to evaluate trial configuration due to error: {e}. Skipping.\n")

print("=== NAS Search Completed! ===")
print(f"Best Config Discovered:\n{best_config}")
print(f"Best Achieved Search Score: {best_score:.4f}")

=== Starting Neural Architecture Search (Trials = 5) ===

Trial 1: Evaluating configuration...
{'depth': 3, 'channels': [32, 32, 32], 'kernels': [3, 3, 3], 'conv_types': ['depthwise_separable', 'standard', 'depthwise_separable'], 'activation': 'leaky_relu'}
-> Results: Params = 14,143 | Validation Acc = 2.18% | Multi-objective Score = 90.2953

Trial 2: Evaluating configuration...
{'depth': 4, 'channels': [64, 16, 128, 32], 'kernels': [5, 3, 3, 3], 'conv_types': ['standard', 'standard', 'standard', 'standard'], 'activation': 'silu'}
-> Results: Params = 73,092 | Validation Acc = 4.44% | Multi-objective Score = 53.4347

Trial 3: Evaluating configuration...
{'depth': 5, 'channels': [16, 32, 16, 128, 64], 'kernels': [5, 5, 3, 5, 5], 'conv_types': ['depthwise_separable', 'depthwise_separable', 'depthwise_separable', 'depthwise_separable', 'standard'], 'activation': 'relu'}
-> Results: Params = 216,095 | Validation Acc = 3.97% | Multi-objective Score = 17.5590

Trial 4: Evaluating configurat

In [12]:
# =====================================================================
# STEP 12: Complete Distillation and Baseline Training for Other Students
# =====================================================================

# In this step, we train our auxiliary models to finalize Question 1 (Requirements 4 & 5)
# Hyperparameters for these benchmarking runs:
# - epochs = 10 (Selected as a balanced choice for stable accuracy trends without excess training time)
# - lr = 1e-3 (standard starting step size for Adam optimizer)
# - distillation_loss_fn = loss_distill_config (combines student and teacher soft logits)

print("=== Training SqueezeNet: Baseline (No Distillation) ===")
squeezenet_baseline = get_squeezenet_student().to(device)
squeezenet_baseline = train_standard_model(
    model=squeezenet_baseline,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: 10 epochs
    lr=1e-3,                      # Hyperparameter: Learning rate/step size
    device=device
)

print("\n=== Training SqueezeNet: Distilled ===")
squeezenet_distilled = get_squeezenet_student().to(device)
squeezenet_distilled = train_distilled_student(
    student=squeezenet_distilled,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: 10 epochs
    lr=1e-3,                      # Hyperparameter: Optimizer learning rate
    device=device,
    distillation_loss_fn=loss_distill_config # Configured loss using T=4.0, alpha=0.3, beta=0.7
)

print("\n=== Training EfficientNet-Lite0: Baseline (No Distillation) ===")
efficientnet_baseline = get_efficientnet_lite0_student().to(device)
efficientnet_baseline = train_standard_model(
    model=efficientnet_baseline,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: 10 epochs
    lr=1e-3,
    device=device
)

print("\n=== Training EfficientNet-Lite0: Distilled ===")
efficientnet_distilled = get_efficientnet_lite0_student().to(device)
efficientnet_distilled = train_distilled_student(
    student=efficientnet_distilled,
    teacher=teacher,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: 10 epochs
    lr=1e-3,
    device=device,
    distillation_loss_fn=loss_distill_config
)

=== Training SqueezeNet: Baseline (No Distillation) ===
--- Starting Training on cuda ---


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/10] (14.5s) - Loss: 4.5758 - Train Acc: 1.66% - Val Acc: 2.49%
Epoch [2/10] (14.2s) - Loss: 4.4077 - Train Acc: 3.53% - Val Acc: 4.41%
Epoch [3/10] (14.5s) - Loss: 4.2121 - Train Acc: 5.51% - Val Acc: 7.20%
Epoch [4/10] (14.3s) - Loss: 4.0504 - Train Acc: 7.31% - Val Acc: 8.12%
Epoch [5/10] (14.4s) - Loss: 3.9368 - Train Acc: 8.88% - Val Acc: 9.56%
Epoch [6/10] (14.2s) - Loss: 3.8527 - Train Acc: 10.47% - Val Acc: 12.37%
Epoch [7/10] (14.3s) - Loss: 3.7703 - Train Acc: 12.01% - Val Acc: 13.79%
Epoch [8/10] (14.4s) - Loss: 3.6965 - Train Acc: 13.42% - Val Acc: 15.39%
Epoch [9/10] (14.4s) - Loss: 3.6377 - Train Acc: 14.45% - Val Acc: 15.86%
Epoch [10/10] (14.1s) - Loss: 3.6150 - Train Acc: 14.98% - Val Acc: 16.09%

=== Training SqueezeNet: Distilled ===
--- Starting Distillation Training on cuda ---
Epoch [1/10] (69.9s) - Distill Loss: 5.6549 - Train Acc: 4.10% - Val Acc: 7.35%
Epoch [2/10] (70.6s) - Distill Loss: 4.2859 - Train Acc: 10.07% - Val Acc: 14.05%
Epoch [3/10] (70.5s)

In [15]:
# =====================================================================
# STEP 13: Fully Training the Discovered Champion NAS Architecture
# =====================================================================

print("=== Training Best Discovered NAS Architecture ===")
print(f"Best Config: {best_config}")

# 1. Instantiate the best discovered architecture configuration mapping
# Parameters in best_config dictate convolutional layers, channel depth, and activations
nas_champion_model = SearchableNet(best_config).to(device)
print(f"Total NAS Model Parameters: {count_parameters(nas_champion_model):,}")

# 2. Train the model on the full training set for 10 epochs to maximize performance
# Hyperparameters for final confirmation:
# - epochs = 10 (Ensures proper convergence while keeping evaluation quick)
# - lr = 1e-3 (Balanced optimizer speed selection)
nas_champion_model = train_standard_model(
    model=nas_champion_model,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,                     # Hyperparameter: Training epochs configured to 10
    lr=1e-3,                      # Hyperparameter: Learning rate
    device=device
)

# 3. Final evaluation summary
final_nas_acc = evaluate_accuracy(nas_champion_model, test_loader, device)
print(f"\n=== Final NAS Model Performance on CIFAR-100 Validation Set: {final_nas_acc:.2f}% ===")

=== Training Best Discovered NAS Architecture ===
Best Config: {'depth': 4, 'channels': [16, 64, 32, 16], 'kernels': [3, 3, 5, 3], 'conv_types': ['depthwise_separable', 'standard', 'depthwise_separable', 'standard'], 'activation': 'leaky_relu'}
Total NAS Model Parameters: 19,503
--- Starting Training on cuda ---
Epoch [1/10] (13.6s) - Loss: 4.3110 - Train Acc: 5.71% - Val Acc: 9.18%
Epoch [2/10] (13.3s) - Loss: 3.8767 - Train Acc: 10.92% - Val Acc: 12.01%
Epoch [3/10] (13.4s) - Loss: 3.6464 - Train Acc: 14.11% - Val Acc: 14.05%
Epoch [4/10] (13.5s) - Loss: 3.4975 - Train Acc: 16.20% - Val Acc: 15.77%
Epoch [5/10] (13.5s) - Loss: 3.3982 - Train Acc: 18.01% - Val Acc: 17.96%
Epoch [6/10] (13.4s) - Loss: 3.3229 - Train Acc: 19.21% - Val Acc: 18.69%
Epoch [7/10] (13.4s) - Loss: 3.2695 - Train Acc: 20.21% - Val Acc: 19.28%
Epoch [8/10] (13.4s) - Loss: 3.2327 - Train Acc: 20.92% - Val Acc: 20.36%
Epoch [9/10] (13.4s) - Loss: 3.2060 - Train Acc: 21.49% - Val Acc: 20.45%
Epoch [10/10] (13.5s) 

## Deliverable Completion Cell

Run this cell after the recorded training cells to save all model checkpoints, export all student and NAS artifacts, and generate benchmark CSV/JSON files.

In [ ]:
# Save all independently trained PyTorch checkpoints.
checkpoint_map = {
    'resnet50_teacher': teacher,
    'mobilenet_v2_baseline': mobilenet_baseline,
    'mobilenet_v2_distilled': mobilenet_distilled,
    'squeezenet_baseline': squeezenet_baseline,
    'squeezenet_distilled': squeezenet_distilled,
    'efficientnet_lite0_baseline': efficientnet_baseline,
    'efficientnet_lite0_distilled': efficientnet_distilled,
    'nas_champion': nas_champion_model,
}
import os
os.makedirs('checkpoints', exist_ok=True)
for name, model in checkpoint_map.items():
    torch.save(model.state_dict(), f'checkpoints/{name}.pth')
    print(f'Saved checkpoints/{name}.pth')

# Export every student and the NAS champion to ONNX.
export_map = {
    'mobilenet_v2_baseline': mobilenet_baseline,
    'mobilenet_v2_distilled': mobilenet_distilled,
    'squeezenet_baseline': squeezenet_baseline,
    'squeezenet_distilled': squeezenet_distilled,
    'efficientnet_lite0_baseline': efficientnet_baseline,
    'efficientnet_lite0_distilled': efficientnet_distilled,
    'nas_champion': nas_champion_model,
}
os.makedirs('onnx', exist_ok=True)
for name, model in export_map.items():
    export_student_to_onnx(model, f'onnx/{name}.onnx')

# Save NAS history and the champion recipe.
import csv, json
with open('nas_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['config', 'params', 'accuracy', 'score'])
    writer.writeheader()
    writer.writerows(nas_results)
with open('nas_champion.json', 'w') as f:
    json.dump({'config': best_config, 'score': best_score, 'validation_accuracy': final_nas_acc}, f, indent=2)

# CPU benchmark with ONNX Runtime, storage size, and peak-process-memory support.
import time
import numpy as np
import psutil
onnx_benchmark_rows = []
dummy = np.random.randn(1, 3, 32, 32).astype(np.float32)
for name in export_map:
    path = f'onnx/{name}.onnx'
    session = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    for _ in range(10):
        session.run(None, {input_name: dummy})
    timings = []
    process = psutil.Process()
    rss_before = process.memory_info().rss
    for _ in range(100):
        start = time.perf_counter()
        session.run(None, {input_name: dummy})
        timings.append((time.perf_counter() - start) * 1000)
    rss_after = process.memory_info().rss
    size_bytes = os.path.getsize(path)
    mean_ms = sum(timings) / len(timings)
    onnx_benchmark_rows.append({
        'model': name,
        'provider': 'CPUExecutionProvider',
        'batch_size': 1,
        'mean_latency_ms': round(mean_ms, 3),
        'throughput_images_per_sec': round(1000 / mean_ms, 2),
        'model_size_bytes': size_bytes,
        'peak_memory_delta_bytes': max(0, rss_after - rss_before),
    })

with open('benchmark_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=onnx_benchmark_rows[0].keys())
    writer.writeheader()
    writer.writerows(onnx_benchmark_rows)

print('Saved checkpoints, ONNX exports, NAS history, and CPU benchmark CSV.')